# No Match Analysis
Created On: 7/26/2025

Created By: Juli Hirt

The objective of this notebook is to generate an analysis of "No Match" user provided terms. This is done over the following steps:
* Column definitions for raw data table
* Display 1: Raw Data Table
* Display 2: Term v Category Rank
* Display 3: Term v Stem Rank

## Imports

In [1]:
import pandas as pd
import numpy as np 
import ast
import string
import json
import polars as pl

## Load Data

In [2]:
file_name = 'cleaned_terms.xlsx'
sheet_name = 'Cleaned Terms'
df = pd.read_excel(f'Data/Outbound/{file_name}', sheet_name=sheet_name)


In [3]:
df.head(3)

,metadata.ResponseId,metadata.Finished,metadata.Progress,metadata.TermTrack,metadata.QuestionId,QX_X.UserProvidedMoodTerm,QX_X.CleanedMoodTerm,QX_X.CleanedTermStem,QX_X.UserCategoryMapping,QX.MultipleCategoryTerm,QX#1_X.MultipleCategoryMapping,QX#2_X_1.UserProvidedCategory,QX#2_X_1.CleanedCategory
0,R_1adu8XGo0BEYT2V,True,100,Recent,Q2_5,Baldurs Gate 3 - fulfilling,fulfilling,fulfil,Adventurous,True,"['Adventurous', 'Dark', 'Horror', 'Humorous']",NaN,NaN
1,R_1adu8XGo0BEYT2V,True,100,Avoid,Q10_2,Boredom,boring,bore,No match,False,NaN,NaN,NaN
2,R_1adu8XGo0BEYT2V,True,100,Recent,Q2_1,"Celeste - thrilling,",thrilling,thrill,Adventurous,True,"['Adventurous', 'Humorous', 'Intense', 'Sensual']",NaN,NaN


In [4]:
matched_df = df[df['QX_X.UserCategoryMapping'] != 'No match'].reset_index(drop=True)
no_match_df = df[df['QX_X.UserCategoryMapping'] == 'No match'].reset_index(drop=True)  

In [5]:
df.shape[0] - matched_df.shape[0] - no_match_df.shape[0]

0

## Display 1

| Column                         | Valid Values                     | Origin    | Definition                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
|--------------------------------|----------------------------------|-----------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| metadata.ResponseId            | String                           | Qualtrics | The ResponseId is an auto-generated unique identifier created by Qualtrics when a participant begins a survey.                                                                                                                                                                                                                                                                                                                                                                                                     |
| metadata.Finished              | Bool (T/F)                       | Qualtrics | The Finished indicator serves to record if a user has completed the entire survey. When the field is 'True' the survey is complete, when it is 'False' the survey is incomplete. This field is created by Qualtrics.                                                                                                                                                                                                                                                                                               |
| metadata.Progress              | Integer                          | Qualtrics | The Progress field represents the completed percentage of the survey. This field is used for filtering throughout the term identification process. <br>Users that have completed at least 33% of the survey have answered all questions in the Recent track.<br>Users that have completed at least 57% of the survey have answered all questions in the Recent and Favorite tracks.<br>Users that have completed at least 100% of the survey have answered all questions in the Recent, Favorite and Avoid tracks. |
| metadata.TermTrack             | String (Recent, Favorite, Avoid) | Analysis  | The Term Track represents on which track the term was provided. This field was extrapolated from the QuestionId.                                                                                                                                                                                                                                                                                                                                                                                                   |
| metadata.QuestionId            | String (Q2_X, Q6_X, Q10_X)       | Qualtrics | The QuestionId represents the Qualtrics question that the term has been mapped to.<br>QuestionIds prefixed with Q2 represent the Recent Track.<br>QuestionIds prefixed with Q6 represent the Favorite track.<br>QuestionIds prefixed with Q10 represent the Avoid track.<br>The X value can range from 1 to 10 and represents the index of the term within the question.                                                                                                                                           |
| QX_X.UserProvidedMoodTerm      | String                           | Qualtrics | The term value provided by the participant for a given track and index.                                                                                                                                                                                                                                                                                                                                                                                                                                            |
| QX_X.CleanedMoodTerm           | String                           | Analysis  | A ""cleaned"" version of the term value provided by the participant.<br>Cleaning steps include:<br>1. Stripping '#'<br>2. Lowercase all text<br>3. Spell check using the Python Spellchecker library                                                                                                                                                                                                                                                                                                               |
| QX_X.CleanedTermStem           | String                           | Analysis  | The stemmed representation of the value generated in Q                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| QX_X.UserCategoryMapping       | String                           | Qualtrics | The GAMER lab category assigned to the participant provided mood term.                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| QX.MultipleCategoryTerm        | Bool (T/F)                       | Analysis  | If the user indicated that for the row term, there are multiple applicable categories which can be assigned.                                                                                                                                                                                                                                                                                                                                                                                                       |
| QX#1_X.MultipleCategoryMapping | Array.String                     | Analysis  | An array of assigned categories for the given row term.                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
| QX#2_X_1.UserProvidedCategory  | String                           | Qualtrics | The participant generated category value for a given term.                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| QX#2_X_1.CleanedCategory       | String                           | Analysis  | A ""cleaned"" version of the category value provided by the participant.<br>Cleaning steps include:<br>1. Lowercase all text<br>2. Spell check using the Python Spellchecker library                                                                                                                                                                                                                                                                                                                               |


In [6]:
def_array = [
  {
    "Column": "metadata.ResponseId",
    "Valid Values": "String",
    "Origin": "Qualtrics",
    "Definition": "The ResponseId is an auto-generated unique identifier created by Qualtrics when a participant begins a survey."
  },
  {
    "Column": "metadata.Finished",
    "Valid Values": "Bool (T/F)",
    "Origin": "Qualtrics",
    "Definition": "The Finished indicator serves to record if a user has completed the entire survey. When the field is 'True' the survey is complete, when it is 'False' the survey is incomplete. This field is created by Qualtrics."
  },
  {
    "Column": "metadata.Progress",
    "Valid Values": "Integer ",
    "Origin": "Qualtrics",
    "Definition": "The Progress field represents the completed percentage of the survey. This field is used for filtering throughout the term identification process. \nUsers that have completed at least 33% of the survey have answered all questions in the Recent track.\nUsers that have completed at least 57% of the survey have answered all questions in the Recent and Favorite tracks.\nUsers that have completed at least 100% of the survey have answered all questions in the Recent, Favorite and Avoid tracks."
  },
  {
    "Column": "metadata.TermTrack",
    "Valid Values": "String (Recent, Favorite, Avoid)",
    "Origin": "Analysis",
    "Definition": "The Term Track represents on which track the term was provided. This field was extrapolated from the QuestionId."
  },
  {
    "Column": "metadata.QuestionId",
    "Valid Values": "String (Q2_X, Q6_X, Q10_X)",
    "Origin": "Qualtrics",
    "Definition": "The QuestionId represents the Qualtrics question that the term has been mapped to.\nQuestionIds prefixed with Q2 represent the Recent Track.\nQuestionIds prefixed with Q6 represent the Favorite track.\nQuestionIds prefixed with Q10 represent the Avoid track.\nThe X value can range from 1 to 10 and represents the index of the term within the question."
  },
  {
    "Column": "QX_X.UserProvidedMoodTerm",
    "Valid Values": "String",
    "Origin": "Qualtrics",
    "Definition": "The term value provided by the participant for a given track and index."
  },
  {
    "Column": "QX_X.CleanedMoodTerm",
    "Valid Values": "String",
    "Origin": "Analysis",
    "Definition": "A \"\"cleaned\"\" version of the term value provided by the participant.\nCleaning steps include:\n1. Stripping '#'\n2. Lowercase all text\n3. Spell check using the Python Spellchecker library"
  },
  {
    "Column": "QX_X.CleanedTermStem",
    "Valid Values": "String",
    "Origin": "Analysis",
    "Definition": "The stemmed representation of the value generated in QX_X.CleanedMoodTerm"
  },
  {
    "Column": "QX_X.UserCategoryMapping",
    "Valid Values": "String",
    "Origin": "Qualtrics",
    "Definition": "The GAMER lab category assigned to the participant provided mood term."
  },
  {
    "Column": "QX.MultipleCategoryTerm",
    "Valid Values": "Bool (T/F)",
    "Origin": "Analysis",
    "Definition": "If the user indicated that for the row term, there are multiple applicable categories which can be assigned."
  },
  {
    "Column": "QX#1_X.MultipleCategoryMapping",
    "Valid Values": "Array.String",
    "Origin": "Analysis",
    "Definition": "An array of assigned categories for the given row term."
  },
  {
    "Column": "QX#2_X_1.UserProvidedCategory",
    "Valid Values": "String",
    "Origin": "Qualtrics",
    "Definition": "The participant generated category value for a given term."
  },
  {
    "Column": "QX#2_X_1.CleanedCategory",
    "Valid Values": "String",
    "Origin": "Analysis",
    "Definition": "A \"\"cleaned\"\" version of the category value provided by the participant.\nCleaning steps include:\n1. Lowercase all text\n2. Spell check using the Python Spellchecker library"
  }
]

def_array_df = pd.DataFrame(def_array)
def_array_df.head(1)

,Column,Valid Values,Origin,Definition
0,metadata.ResponseId,String,Qualtrics,The ResponseId is an auto-generated unique ide...


In [7]:
dis1_df = no_match_df.copy()

## Display 2

### Display 2a

In [8]:
# convert the QX#1_X.MultipleCategoryMapping column from string to list
no_match_df['QX#1_X.MultipleCategoryMapping'] = no_match_df['QX#1_X.MultipleCategoryMapping'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [9]:
no_match_df.dtypes

metadata.ResponseId               object
metadata.Finished                   bool
metadata.Progress                  int64
metadata.TermTrack                object
metadata.QuestionId               object
QX_X.UserProvidedMoodTerm         object
QX_X.CleanedMoodTerm              object
QX_X.CleanedTermStem              object
QX_X.UserCategoryMapping          object
QX.MultipleCategoryTerm             bool
QX#1_X.MultipleCategoryMapping    object
QX#2_X_1.UserProvidedCategory     object
QX#2_X_1.CleanedCategory          object
dtype: object

In [10]:
# gather all categories for the study
categories = df['QX_X.UserCategoryMapping'].unique().tolist()

#remove whitespace
categories = [cat.strip() for cat in categories if isinstance(cat, str)]

#sort array alphabetically
categories.sort()

categories[categories.index('No match')] = 'No Category'
categories

['Adventurous',
 'Aggressive',
 'Cute',
 'Dark',
 'Horror',
 'Humorous',
 'Imaginative',
 'Intense',
 'Light-hearted',
 'Mysterious',
 'No Category',
 'Peaceful',
 'Quirky',
 'Romantic',
 'Sad',
 'Sarcastic',
 'Sensual',
 'Solitary']

In [11]:
import pandas as pd
import string

# -------------------
# Step 1: Process and group terms
# -------------------

term_summary = {}

for idx, row in no_match_df.iterrows():
    text = row['QX_X.CleanedMoodTerm'].strip()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()

    # Use pre-cleaned stem
    stem = row['QX_X.CleanedTermStem'].strip()

    # Handle category extraction robustly
    if isinstance(row['QX#1_X.MultipleCategoryMapping'], list):
        row_categories = row['QX#1_X.MultipleCategoryMapping']
    elif pd.isna(row['QX#1_X.MultipleCategoryMapping']):
        row_categories = ['No Category']
    else:
        try:
            parsed = ast.literal_eval(row['QX#1_X.MultipleCategoryMapping'])
            row_categories = parsed if isinstance(parsed, list) else ['No Category']
        except:
            row_categories = ['No Category']

    row_categories = list(set(row_categories))

    if text not in term_summary:
        term_summary[text] = {
            'Stem': stem,
            'Categories': row_categories,
            'Count': 1,
            'CategoryCounts': {cat: 1 if cat in row_categories else 0 for cat in categories}
        }
    else:
        term_summary[text]['Count'] += 1
        for cat in row_categories:
            term_summary[text]['CategoryCounts'][cat] = term_summary[text]['CategoryCounts'].get(cat, 0) + 1

# Ensure all category counts exist for all terms
for term_info in term_summary.values():
    for cat in categories:
        term_info['CategoryCounts'].setdefault(cat, 0)

# -------------------
# Step 2: Build term-category matrix (binary indicators)
# -------------------

cat_dict = {
    term: {cat: 1 if cat in info['Categories'] else 0 for cat in categories}
    for term, info in term_summary.items()
}

df_terms = pd.DataFrame.from_dict(cat_dict, orient='index')
df_terms = df_terms.fillna(0).astype(int)

# -------------------
# Step 3: Build stem-to-terms mapping
# -------------------

stem_to_terms = {}
for term, info in term_summary.items():
    stem = info['Stem']
    stem_to_terms.setdefault(stem, []).append(term)

# -------------------
# Step 4: Build grouped DataFrame with header rows and correct counts
# -------------------

new_rows = []

for stem in sorted(stem_to_terms.keys()):
    terms = stem_to_terms[stem]
    stem_count = sum(term_summary[term]['Count'] for term in terms)

    totals = pd.Series(0, index=categories)
    for term in terms:
        for col in categories:
            totals[col] += term_summary[term]['CategoryCounts'][col]

    header_row = {'Term': f"Stem: {stem}", 'RowType': 'Stem', 'Count': stem_count}
    header_row.update(totals.to_dict())
    new_rows.append(header_row)

    for term in terms:
        row_data = {
            'Term': f"   → {term}",
            'RowType': 'Term',
            'Count': term_summary[term]['Count']
        }
        row_data.update(term_summary[term]['CategoryCounts'])
        new_rows.append(row_data)

# -------------------
# Step 5: Create final DataFrame and display
# -------------------

df_grouped = pd.DataFrame(new_rows)

cols = ['Term', 'RowType', 'Count'] + categories
df_grouped = df_grouped[cols]

display(df_grouped)


,Term,RowType,Count,Adventurous,Aggressive,Cute,Dark,Horror,Humorous,Imaginative,...,Light-hearted,Mysterious,No Category,Peaceful,Quirky,Romantic,Sad,Sarcastic,Sensual,Solitary
0,Stem: agil,Stem,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,→ agile,Term,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,Stem: angri,Stem,3,0,0,0,0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
3,→ angry,Term,3,0,0,0,0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
4,Stem: annoy,Stem,2,0,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,→ unpredictable,Term,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
185,Stem: unstabl,Stem,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
186,→ unstable,Term,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
187,Stem: vulgar,Stem,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


## Create Report


In [12]:
with pd.ExcelWriter('Data/Outbound/no_match_terms.xlsx') as writer:
    def_array_df.to_excel(writer, sheet_name='Column Definitions', index=False)
    dis1_df.to_excel(writer, sheet_name='Display 1', index=False)
    df_grouped.to_excel(writer, sheet_name='Display 2', index=False)